# Swahili News Topic Classification: AfroXLMR

Model 3 from `swahili_news_baseline.ipynb`. Same cleaning, same stratified split (`random_state=42`), same metrics. The classifier is `Davlan/afro-xlmr-base`, an XLM-RoBERTa model adapted to African languages, including Swahili (Alabi et al., 2022).

The official test set is scored once, after the checkpoint is chosen on validation macro-F1.

Run this notebook on Google Colab with a GPU. Use Runtime → Change runtime type → T4 GPU, then Run all. The session downloads the pretrained model (about 1.1 GB). An 8 GB laptop is too small for this fine-tune. Section 11 downloads the metrics and the error sheet. The model folder itself is about 1 GB, so copy that to Google Drive if you need to keep it: Colab deletes the session disk when the runtime stops.


## 1. Setup

Colab already includes a CUDA build of PyTorch. Leave that build in place. A fresh `pip install torch` on Colab can replace it with a CPU wheel, and then the GPU sits idle.

If the training cell runs out of GPU memory, set `MAX_LEN = 128` and `BATCH_SIZE = 4` in the next cell and rerun from the top.


In [ ]:
# Do not install torch here. Colab's preinstalled build is the CUDA one.
%pip install -q transformers accelerate datasets scikit-learn pandas matplotlib seaborn sentencepiece


In [ ]:
%matplotlib inline
import json
import random
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

SEED = 42
MODEL_NAME = "Davlan/afro-xlmr-base"
MAX_LEN = 256          # same truncation as the BiGRU notebook
BATCH_SIZE = 8
GRAD_ACCUM = 2         # effective batch size 16
EPOCHS = 3
PATIENCE = 2
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.1
# Paths are relative to the repo root, whether this runs from notebooks/ or from Colab's /content.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODELS_DIR = ROOT / "models"
RESULTS_DIR = ROOT / "results" / "afroxlmr"
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR = MODELS_DIR / "swahili_afroxlmr"
METRICS_PATH = RESULTS_DIR / "afroxlmr_metrics.json"
HISTORY_PATH = RESULTS_DIR / "afroxlmr_history.json"
ERRORS_PATH = RESULTS_DIR / "afroxlmr_errors.csv"

def mps_available():
    backend = getattr(torch.backends, "mps", None)
    return backend is not None and backend.is_available()

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if mps_available():
        torch.mps.manual_seed(seed)

set_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Device: cuda |", torch.cuda.get_device_name(0))
elif mps_available():
    device = torch.device("mps")
    print("Device: mps")
    print("This fine-tune is sized for a Colab T4. On an 8 GB Mac it is likely to run out of memory.")
else:
    device = torch.device("cpu")
    print("Device: cpu")
    print("No GPU detected. In Colab: Runtime → Change runtime type → T4 GPU, then rerun this notebook from the top.")

use_amp = device.type == "cuda"
print("Working directory:", Path.cwd())
print("Mixed precision:", use_amp)


## 2. Load the data

Same source as the baseline. The official test split stays untouched, and the validation set is cut from train.


In [ ]:
try:
    ds = load_dataset("community-datasets/swahili_news")
except Exception as e:
    print("First attempt failed:", e)
    ds = load_dataset("swahili_news")

print(ds)
label_names = ds["train"].features["label"].names
print("Classes:", label_names)


In [ ]:
train_full = ds["train"].to_pandas()
test_df = ds["test"].to_pandas()

train_full["label_name"] = train_full["label"].map(lambda i: label_names[i])
test_df["label_name"] = test_df["label"].map(lambda i: label_names[i])

train_full.head()


## 3. Preprocessing

Same cleaner as the baseline and the BiGRU notebook. Duplicates and empty strings are dropped from train only.


In [ ]:
URL_RE   = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+")
NONWORD  = re.compile(r"[^a-zA-Z\u00C0-\u017F'\s]")
SPACES   = re.compile(r"\s+")

def clean_text(t: str) -> str:
    t = t.lower()
    t = URL_RE.sub(" ", t)
    t = EMAIL_RE.sub(" ", t)
    t = NONWORD.sub(" ", t)
    t = SPACES.sub(" ", t).strip()
    return t

for df in (train_full, test_df):
    df["clean"] = df["text"].apply(clean_text)

before = len(train_full)
train_full = train_full[train_full["clean"].str.len() > 0].drop_duplicates(subset="clean").reset_index(drop=True)
print(f"Train: {before} -> {len(train_full)} after removing empties/duplicates")

leak = set(train_full["clean"]) & set(test_df["clean"])
print("Texts appearing in both train and test:", len(leak))


## 4. Splits

Same 90/10 stratified cut of the cleaned training set, seed 42. The official test set is not used for fitting or for choosing the checkpoint.


In [ ]:
train_df, val_df = train_test_split(
    train_full, test_size=0.10, stratify=train_full["label"], random_state=SEED
)
train_df, val_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}")
pd.DataFrame({
    "train": train_df["label_name"].value_counts(normalize=True),
    "val": val_df["label_name"].value_counts(normalize=True),
    "test": test_df["label_name"].value_counts(normalize=True),
}).round(3)


## 5. Tokeniser, model, and loss

AfroXLMR-base has about 278 million parameters. Sequences are truncated to 256 wordpieces, the same cap the BiGRU used. The loss is class-weighted cross-entropy, with weights `n / (k * count)`, matching `class_weight="balanced"` on the logistic regression baseline.

Loading the model prints a warning that the classification layer is newly initialised. That layer did not exist in the pretrained checkpoint, so the warning is expected. The embedding and encoder are fine-tuned, not frozen. AdamW uses a smaller learning rate than the BiGRU because these weights are already trained. Mixed precision and gradient checkpointing keep the fine-tune inside a T4's memory.


In [ ]:
class EncodedNews(Dataset):
    def __init__(self, frame, tokenizer):
        enc = tokenizer(
            frame["clean"].tolist(),
            truncation=True,
            max_length=MAX_LEN,
            padding=False,
        )
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels = frame["label"].astype(int).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {
            "input_ids": self.input_ids[i],
            "attention_mask": self.attention_mask[i],
            "labels": self.labels[i],
        }

def collate(features):
    labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)
    features = [{k: v for k, v in f.items() if k != "labels"} for f in features]
    batch = tokenizer.pad(features, return_tensors="pt")
    batch["labels"] = labels
    return batch

set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_ds = EncodedNews(train_df, tokenizer)
val_ds = EncodedNews(val_df, tokenizer)
test_ds = EncodedNews(test_df, tokenizer)

lengths = np.array([len(ids) for ids in train_ds.input_ids])
print(
    f"Train wordpieces: median {np.median(lengths):.0f}, "
    f"95th {np.percentile(lengths, 95):.0f}, "
    f"share truncated to {MAX_LEN}: {(lengths >= MAX_LEN).mean():.1%}"
)

pin = device.type == "cuda"
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate,
    generator=torch.Generator().manual_seed(SEED),
    pin_memory=pin,
)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collate, pin_memory=pin)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collate, pin_memory=pin)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label={i: name for i, name in enumerate(label_names)},
    label2id={name: i for i, name in enumerate(label_names)},
)
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.to(device)

y_train = train_df["label"].to_numpy()
class_counts = np.bincount(y_train, minlength=len(label_names))
class_weights = torch.tensor(
    len(y_train) / (len(label_names) * class_counts), dtype=torch.float32, device=device
)
print("Class weights:", {label_names[i]: round(float(class_weights[i]), 3) for i in range(len(label_names))})

decay, no_decay = [], []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if param.ndim == 1 or name.endswith(".bias"):
        no_decay.append(param)
    else:
        decay.append(param)
optimizer = torch.optim.AdamW(
    [
        {"params": decay, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay, "weight_decay": 0.0},
    ],
    lr=LR,
)
updates_per_epoch = (len(train_loader) + GRAD_ACCUM - 1) // GRAD_ACCUM
total_updates = updates_per_epoch * EPOCHS
warmup_steps = max(1, int(WARMUP_FRAC * total_updates))
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_updates)
try:
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
except TypeError:
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")
print(f"Optimizer steps: {total_updates} ({updates_per_epoch} per epoch, warmup {warmup_steps})")


## 6. Training

Each epoch is scored on the validation set. The checkpoint on disk is replaced whenever validation macro-F1 improves, so a Colab disconnect keeps the best weights seen so far. Training stops early if validation macro-F1 fails to improve for two epochs.


In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    total_loss, seen = 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    for step, batch in enumerate(loader, start=1):
        labels = batch.pop("labels").to(device)
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(**batch).logits
        loss = F.cross_entropy(logits.float(), labels, weight=class_weights)
        scaled = loss / GRAD_ACCUM
        if use_amp:
            scaler.scale(scaled).backward()
        else:
            scaled.backward()
        total_loss += loss.item() * labels.size(0)
        seen += labels.size(0)
        if step % GRAD_ACCUM == 0 or step == len(loader):
            if use_amp:
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if use_amp:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
        if step == 1 or step % 100 == 0 or step == len(loader):
            print(f"  batch {step}/{len(loader)} loss={loss.item():.4f}", flush=True)
    return total_loss / max(seen, 1)

def predict_proba(model, loader):
    model.eval()
    preds, probs = [], []
    with torch.no_grad():
        for batch in loader:
            labels = batch.pop("labels")
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            logits = model(**batch).logits.float()
            prob = torch.softmax(logits, dim=-1).cpu().numpy()
            probs.append(prob)
            preds.append(prob.argmax(axis=1))
            del labels
    return np.concatenate(preds), np.concatenate(probs)

def score(labels, preds, name):
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    print(f"[{name}] accuracy={acc:.4f}  macro-P={p:.4f}  macro-R={r:.4f}  macro-F1={f1:.4f}")
    return {"accuracy": float(acc), "macro_p": float(p), "macro_r": float(r), "macro_f1": float(f1)}

def save_checkpoint(model, tokenizer, epoch):
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(CKPT_DIR)
    tokenizer.save_pretrained(CKPT_DIR)
    (CKPT_DIR / "best_epoch.txt").write_text(str(epoch))

set_seed(SEED)
history = []
best_f1 = -1.0
best_epoch = 0
wait = 0

t0 = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler)
    print(f"[epoch {epoch}] train_loss={train_loss:.4f}", flush=True)
    val_preds, _val_proba = predict_proba(model, val_loader)
    metrics = score(val_df["label"].to_numpy(), val_preds, f"epoch {epoch} validation")
    history.append({"epoch": epoch, "train_loss": train_loss, **metrics})
    HISTORY_PATH.write_text(json.dumps(history, indent=2))

    if metrics["macro_f1"] > best_f1:
        best_f1 = metrics["macro_f1"]
        best_epoch = epoch
        save_checkpoint(model, tokenizer, best_epoch)
        print(f"  saved {CKPT_DIR} (epoch {best_epoch})", flush=True)
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stop at epoch {epoch}. Best epoch is {best_epoch} (val macro-F1={best_f1:.4f}).")
            break

train_seconds = time.perf_counter() - t0
model = AutoModelForSequenceClassification.from_pretrained(CKPT_DIR).to(device)
print(f"Restored epoch {best_epoch}. Training time: {train_seconds / 60:.1f} min")


In [ ]:
hist = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(hist["epoch"], hist["train_loss"], marker="o")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Train loss"); axes[0].set_title("Training loss")
axes[1].plot(hist["epoch"], hist["macro_f1"], marker="o")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Macro-F1"); axes[1].set_title("Validation macro-F1")
plt.tight_layout(); plt.show()


## 7. Evaluation

Validation first, on the restored checkpoint. The test set is scored once.


In [ ]:
val_preds, val_proba = predict_proba(model, val_loader)
val_metrics = score(val_df["label"].to_numpy(), val_preds, "validation")


In [ ]:
test_preds, test_proba = predict_proba(model, test_loader)
test_metrics = score(test_df["label"].to_numpy(), test_preds, "test")
print()
print(classification_report(
    test_df["label"], test_preds, labels=list(range(len(label_names))),
    target_names=label_names, digits=3, zero_division=0,
))


In [ ]:
cm = confusion_matrix(test_df["label"], test_preds, labels=list(range(len(label_names))))
cm_norm = cm / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(7, 6))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("AfroXLMR confusion matrix (row-normalised)")
plt.tight_layout(); plt.show()


## 8. Error analysis

Misclassified test articles, sorted by confidence. The `cause` column is left blank for hand labels.


In [ ]:
errors = test_df.copy()
errors["pred"] = [label_names[i] for i in test_preds]
errors["confidence"] = test_proba.max(axis=1)
errors = errors[errors["label"] != test_preds].sort_values("confidence", ascending=False)
errors["cause"] = ""

print(f"{len(errors)} misclassified out of {len(test_df)} ({len(errors) / len(test_df):.1%})")
pd.set_option("display.max_colwidth", 300)
errors[["label_name", "pred", "confidence", "text"]].head(15)


In [ ]:
pairs = errors.groupby(["label_name", "pred"]).size().sort_values(ascending=False).head(10)
pairs


In [ ]:
errors.to_csv(ERRORS_PATH, index=False)
print("Wrote", ERRORS_PATH)


## 9. Save metrics

The weights and tokeniser were written to `swahili_afroxlmr/` whenever validation improved. This cell records accuracy, macro-F1, training time, and model size for the comparison table.


In [ ]:
ckpt_bytes = sum(p.stat().st_size for p in CKPT_DIR.rglob("*") if p.is_file())
metrics = {
    "model": "AfroXLMR-base",
    "pretrained": MODEL_NAME,
    "val": {k: round(v, 4) for k, v in val_metrics.items()},
    "test": {k: round(v, 4) for k, v in test_metrics.items()},
    "train_minutes": round(train_seconds / 60, 2),
    "best_epoch": best_epoch,
    "n_params": n_params,
    "checkpoint_mb": round(ckpt_bytes / 1e6, 2),
    "max_len": MAX_LEN,
}
METRICS_PATH.write_text(json.dumps(metrics, indent=2))
print(f"Saved {CKPT_DIR} ({ckpt_bytes / 1e6:.1f} MB, {n_params:,} parameters)")
print(json.dumps(metrics, indent=2))


## 10. Quick try-out

The first sentence is the same one used in the baseline notebook. The second is a price story. The third is a health-ministry warning, which the BiGRU mixed up with national news.


In [ ]:
def predict(text, k=3):
    enc = tokenizer(clean_text(text), truncation=True, max_length=MAX_LEN, return_tensors="pt")
    enc = {key: value.to(device) for key, value in enc.items()}
    model.eval()
    with torch.no_grad():
        p = torch.softmax(model(**enc).logits, dim=-1)[0].cpu().numpy()
    top = np.argsort(p)[::-1][:k]
    return [(label_names[i], round(float(p[i]), 3)) for i in top]

examples = [
    "Timu ya taifa ya mpira wa miguu imeshinda mechi ya jana usiku",
    "Bei ya sukari na mafuta imepanda katika soko la dunia",
    "Wizara ya afya yatoa tahadhari kuhusu mlipuko wa ugonjwa",
]
for text in examples:
    print(text)
    print(" ", predict(text))


## 11. Download the results

The metrics file and the error sheet are small enough to download in the browser. The model directory is about 1 GB. Uncomment the Drive lines if you want that copy kept after the runtime stops.


In [ ]:
import zipfile

bundle = RESULTS_DIR / "afroxlmr_colab_outputs.zip"
names = [METRICS_PATH, ERRORS_PATH, HISTORY_PATH]
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in names:
        if path.exists():
            z.write(path, path.name)
            print(f"  added {path.name}")
print(f"Wrote {bundle.resolve()} ({bundle.stat().st_size / 1e6:.1f} MB)")
print(f"Model directory: {CKPT_DIR.resolve()}")

# from google.colab import drive
# import shutil
# drive.mount("/content/drive")
# shutil.copytree(CKPT_DIR, "/content/drive/MyDrive/swahili_afroxlmr", dirs_exist_ok=True)
# shutil.copy(METRICS_PATH, "/content/drive/MyDrive/afroxlmr_metrics.json")

try:
    from google.colab import files
except ImportError:
    print(f"Local run: the files are in {RESULTS_DIR} and {CKPT_DIR}.")
else:
    files.download(str(bundle))
